In [11]:
import os
import ee
from dotenv import load_dotenv

load_dotenv()
ee.Initialize(project=os.getenv("EE_PROJECT"))

print("Earth Engine initialized with project")

Earth Engine initialized with project


In [9]:
# Rough Accra study area (we'll replace it with official boundaries later)
accra = ee.Geometry.Rectangle([-0.35, 5.45, 0.05, 5.80])

def get_collection(collection_id):
    return (ee.ImageCollection(collection_id)
            .filterBounds(accra)
            .filterDate("2023-01-01", "2023-12-31")
            .filter(ee.Filter.lt("CLOUD_COVER", 30)))

l8 = get_collection("LANDSAT/LC08/C02/T1_L2")
l9 = get_collection("LANDSAT/LC09/C02/T1_L2")

print("Landsat 8 scenes (2023, <30% cloud):", l8.size().getInfo())
print("Landsat 9 scenes (2023, <30% cloud):", l9.size().getInfo())

Landsat 8 scenes (2023, <30% cloud): 7
Landsat 9 scenes (2023, <30% cloud): 7


In [3]:
import pandas as pd

def scene_dates(collection, satellite):
    dates = collection.aggregate_array("system:time_start").getInfo()
    clouds = collection.aggregate_array("CLOUD_COVER").getInfo()
    df = pd.DataFrame({"date": pd.to_datetime(dates, unit="ms"), "cloud_cover": clouds})
    df["satellite"] = satellite
    return df

scenes = pd.concat([scene_dates(l8, "Landsat 8"), scene_dates(l9, "Landsat 9")])
scenes["month"] = scenes["date"].dt.month

print(scenes.sort_values("date").to_string(index=False))
print()
print(scenes.groupby("month").size())

                   date  cloud_cover satellite  month
2023-01-02 10:16:09.133         4.50 Landsat 9      1
2023-01-10 10:16:04.618         8.02 Landsat 8      1
2023-01-18 10:16:01.351        27.86 Landsat 9      1
2023-01-26 10:16:04.162        26.74 Landsat 8      1
2023-02-03 10:16:11.251        16.49 Landsat 9      2
2023-02-11 10:16:00.698        12.04 Landsat 8      2
2023-03-15 10:15:39.522        16.45 Landsat 8      3
2023-04-16 10:15:14.465        22.09 Landsat 8      4
2023-05-26 10:15:10.453         9.22 Landsat 9      5
2023-11-02 10:15:49.063        18.37 Landsat 9     11
2023-11-10 10:15:49.515        13.33 Landsat 8     11
2023-12-04 10:15:57.955        19.08 Landsat 9     12
2023-12-12 10:15:51.487        11.55 Landsat 8     12
2023-12-20 10:16:02.279         2.67 Landsat 9     12

month
1     4
2     2
3     1
4     1
5     1
11    2
12    3
dtype: int64


In [4]:
def count_by_month(collection_id, start, end, max_cloud):
    col = (ee.ImageCollection(collection_id)
           .filterBounds(accra)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUD_COVER", max_cloud)))
    dates = col.aggregate_array("system:time_start").getInfo()
    return pd.to_datetime(dates, unit="ms")

all_dates = []
for cid in ["LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"]:
    all_dates.extend(count_by_month(cid, "2019-01-01", "2024-12-31", 70))

df = pd.DataFrame({"date": all_dates})
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

print(pd.crosstab(df["year"], df["month"]))

month  1   2   3   4   5   6   7   8   9   10  11  12
year                                                 
2019    2   1   1   1   2   2   0   2   0   2   1   2
2020    2   2   1   1   0   1   1   1   0   1   2   2
2021    2   2   1   2   2   2   0   1   0   1   5   3
2022    3   3   3   2   2   2   2   2   0   3   3   4
2023    4   3   2   2   2   0   1   2   0   2   3   4
2024    4   3   2   4   4   2   2   2   1   2   4   3


In [5]:
import geemap

# Pick the Landsat 9 photo from 20 Dec 2023
img = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
       .filterBounds(accra)
       .filterDate("2023-12-20", "2023-12-21")
       .first())

# Convert raw numbers to real reflectance values (standard Landsat Collection 2 scaling)
def scale(image):
    return image.select("SR_B.").multiply(0.0000275).add(-0.2)

rgb = scale(img).select(["SR_B4", "SR_B3", "SR_B2"])  # red, green, blue

m = geemap.Map(center=[5.6, -0.15], zoom=10)
m.addLayer(rgb, {"min": 0.0, "max": 0.3}, "Landsat 9 true colour, 20 Dec 2023")
m.addLayer(accra, {"color": "red"}, "Study area (rough box)")
m

Map(center=[5.6, -0.15], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [6]:
qa = img.select("QA_PIXEL")

cloud = qa.bitwiseAnd(1 << 3).neq(0)    # bit 3 = cloud
shadow = qa.bitwiseAnd(1 << 4).neq(0)   # bit 4 = cloud shadow

m.addLayer(cloud.selfMask(), {"palette": ["red"]}, "Cloud (Landsat flag)")
m.addLayer(shadow.selfMask(), {"palette": ["yellow"]}, "Cloud shadow (Landsat flag)")
m

Map(bottom=127296.0, center=[5.5886511909594905, -0.193634033203125], controls=(WidgetControl(options=['positi…

In [7]:
def mask_clouds(image):
    qa = image.select("QA_PIXEL")
    # Bits: 1 = dilated cloud, 2 = cirrus, 3 = cloud, 4 = cloud shadow
    bad = (qa.bitwiseAnd(1 << 1).neq(0)
           .Or(qa.bitwiseAnd(1 << 2).neq(0))
           .Or(qa.bitwiseAnd(1 << 3).neq(0))
           .Or(qa.bitwiseAnd(1 << 4).neq(0)))
    return image.updateMask(bad.Not())

img_masked = mask_clouds(img)
rgb_masked = scale(img_masked).select(["SR_B4", "SR_B3", "SR_B2"])

m2 = geemap.Map(center=[5.6, -0.15], zoom=10)
m2.addLayer(rgb_masked, {"min": 0.0, "max": 0.3}, "Cloud-masked, 20 Dec 2023")
m2.addLayer(accra, {"color": "red"}, "Study area (rough box)")
m2

Map(center=[5.6, -0.15], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …